# CSFR round 2: the JPEG corruption CSFR solves on Kaggle GPUs

Runs `scripts/run_jpeg_corruption.py --csfr-only` (E9) on CUDA in float64 through
`kaggle/jpeg_gpu_kernel.py`: both CSFR variants for every case and condition of the 105 files,
one shard per GPU. The baselines and LaMa run on the workstation.

* Settings: accelerator **GPU T4 x2**, internet **on** (needs a phone-verified account).
* Run with **Save Version > Save & Run All**: it runs headless for up to 12 h. The driver stops the
  shards before the limit and saves `state/`; an interactive session would stop when idle.
* Session 2 onward (only if `STATUS` says incomplete): **Add Input > Your Work >** this notebook's
  previous version (its output holds `state/`), then Save & Run All again.
* Outputs: `STATUS` (complete / incomplete / failed: reason), `results.tar.gz`, `kaggle_env.json`,
  `shard0.log`, `shard1.log`, and `state/` while incomplete.
* Independent of the downstream notebook: either can run first, or both at once.


In [ ]:
# the code this run uses; kaggle_env.json records the commit it ran.
# A new COMMIT reaches Kaggle only after `git push` AND pasting this cell into the Kaggle editor.
REPO = "https://github.com/yoadjei/csfr.git"
BRANCH = "feat/ieee-access-r2"
COMMIT = "977231fd15997faa05554e93687b2e0bd580c144"

import subprocess
def git(*args):
    return subprocess.run(["git", "-C", "/tmp/paper2_recon", *args], capture_output=True, text=True)

subprocess.run(["rm", "-rf", "/tmp/paper2_recon"], check=True)
subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO, "/tmp/paper2_recon"], check=True)
if git("checkout", "-q", COMMIT).returncode:
    raise SystemExit(f"{COMMIT} is not on GitHub's {BRANCH}: push it, then run again")
# code commits after the pin (notebooks and the paper excluded): a stale pin runs old code
newer = git("log", "--oneline", f"{COMMIT}..origin/{BRANCH}", "--", ".",
            ":(exclude)*.ipynb", ":(exclude)paper").stdout.strip()
if newer:
    print(f"WARNING: {BRANCH} has code commits after the pin; this run does not include:")
    print(newer)
    print()
print(git("log", "-1", "--oneline").stdout)


In [ ]:
!python /tmp/paper2_recon/kaggle/jpeg_gpu_kernel.py


In [ ]:
# STATUS is written on every path the driver controls; without it the driver never ran it
# (or was killed): its exit code and its output in the cell above say why
from pathlib import Path
status = Path("/kaggle/working/STATUS")
if not status.exists():
    raise SystemExit(f"no STATUS: the driver exited with code {_exit_code}; see the output above")
print(status.read_text())
